# 🔌 Agents with Microsoft Learn MCP Server

Welcome to our **Model Context Protocol (MCP)** tutorial! In this notebook, we'll demonstrate how to:

1. **Understand** the MCP standard and how agents connect to remote MCP servers
2. **Configure** the Microsoft Learn MCP Server as an agent tool
3. **Create** an AI agent that can search and retrieve official Microsoft documentation
4. **Query** the agent with questions about Microsoft Foundry models and AI concepts

### What is MCP?
> The [Model Context Protocol](https://modelcontextprotocol.io/) (MCP) is an open standard that defines how applications provide tools and contextual data to large language models. It enables consistent, scalable integration of external tools into model workflows — so an agent can call a search engine, a database, or any external service through a standardised interface.

### The Microsoft Learn MCP Server
Microsoft hosts a public MCP server backed by their official documentation knowledge base — the same service that powers Ask Learn and Copilot for Azure. It exposes three tools:

| Tool | Description |
|------|-------------|
| `microsoft_docs_search` | Semantic search across all Microsoft documentation |
| `microsoft_docs_fetch` | Retrieve a complete documentation article |
| `microsoft_code_sample_search` | Search for official code samples |

**No authentication is required.** This makes it an ideal first MCP server to connect to.

## Prerequisites
- A `.env` file in the parent directory containing:
  ```bash
  AI_FOUNDRY_PROJECT_ENDPOINT=<your-ai-foundry-project-endpoint>
  AZURE_AI_MODEL_DEPLOYMENT_NAME=<supported-model>
  AZURE_CLIENT_ID=<service-principal-app-id>
  AZURE_CLIENT_SECRET=<service-principal-secret>
  AZURE_TENANT_ID=<your-tenant-id>
  ```

## Architecture Overview
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────────────┐
│   User Query    │───▶│  Agent + MCP Tool │───▶│  Microsoft Learn MCP    │
│                 │     │                  │     │  learn.microsoft.com   │
└─────────────────┘     └──────────────────┘     └─────────────────────────┘
                                                            │
                                                            ▼
                                               ┌───────────────────────┐
                                               │  - microsoft_docs_search  │
                                               │  - microsoft_docs_fetch   │
                                               │  - code_sample_search     │
                                               └───────────────────────┘
```

## 1. Initial Setup
We'll load environment variables from `.env` and initialise our **AIProjectClient** to manage agents with MCP tools.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    MCPTool,
)
from openai.types.responses.response_input_param import (
    McpApprovalResponse,
    ResponseInputParam,
)

# Load environment variables from parent .env
notebook_path = Path().absolute()
parent_dir = notebook_path.parent
load_dotenv(parent_dir.parent / '.env')


# Quick configuration + auth smoke test (no interactive login)
required = [
    'AI_FOUNDRY_PROJECT_ENDPOINT',
    'AZURE_AI_MODEL_DEPLOYMENT_NAME',
    'AZURE_CLIENT_ID',
    'AZURE_CLIENT_SECRET',
    'AZURE_TENANT_ID',
]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise ValueError(
        f'Missing required environment variables: {missing}. '
        'Ensure your repo-root .env is populated and loaded.'
    )

try:
    DefaultAzureCredential().get_token('https://management.azure.com/.default')
    print('✅ Auth check passed (can get Azure token)')
except Exception as e:
    raise RuntimeError(
        'Auth check failed. Verify your environment variables are correct and that network access to Azure is available. '
        f'Details: {e}'
    )

# Get configuration
project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

print("🔌 Microsoft Learn MCP Server Notebook")
print("=" * 60)
print(f"📍 Project Endpoint: {project_endpoint}")
print(f"🤖 Model Deployment: {model_deployment}")
print("")
print("🔌 MCP Server: https://learn.microsoft.com/api/mcp")
print("🔓 Authentication: None required (public server)")


## Service Principal Check
Verify that the service principal credentials are configured correctly before proceeding. These variables are loaded from your `.env` file and used by `DefaultAzureCredential` automatically — no `az login` required.

In [ ]:
import os
from azure.identity import ClientSecretCredential

client_id     = os.getenv('AZURE_CLIENT_ID', '')
client_secret = os.getenv('AZURE_CLIENT_SECRET', '')
tenant_id     = os.getenv('AZURE_TENANT_ID', '')

print('Service Principal Configuration')
print('=' * 40)
print(f'AZURE_TENANT_ID:     {tenant_id[:8]}...  ({len(tenant_id)} chars)')
print(f'AZURE_CLIENT_ID:     {client_id[:8]}...  ({len(client_id)} chars)')
print(f'AZURE_CLIENT_SECRET: {"*" * 8}  ({len(client_secret)} chars)')
print()

missing = [k for k, v in {
    'AZURE_TENANT_ID': tenant_id,
    'AZURE_CLIENT_ID': client_id,
    'AZURE_CLIENT_SECRET': client_secret,
}.items() if not v]

if missing:
    raise ValueError(f'Missing service principal variables: {missing}. Check your .env file.')

# Validate credentials by acquiring a token directly with ClientSecretCredential
try:
    sp_credential = ClientSecretCredential(tenant_id, client_id, client_secret)
    sp_credential.get_token('https://management.azure.com/.default')
    print('✅ Service principal credentials are valid — token acquired successfully')
except Exception as e:
    raise RuntimeError(
        f'Service principal authentication failed: {e}\n'
        'Check that AZURE_CLIENT_ID, AZURE_CLIENT_SECRET, and AZURE_TENANT_ID are correct '
        'and that the service principal has not expired.'
    )


## 2. Initialise the AI Project Client
Create the AIProjectClient using `DefaultAzureCredential`, which picks up the environment variables
(`AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_TENANT_ID`) from the environment automatically.
No interactive login step is required.

In [ ]:
# Initialise AIProjectClient with DefaultAzureCredential.
# environment variables are loaded from .env via load_dotenv above.
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential
)

openai_client = project_client.get_openai_client()

print("✅ AIProjectClient initialised with DefaultAzureCredential")
print("✅ OpenAI client ready for conversations")

## 3. Define the Microsoft Learn MCP Tool

The **Microsoft Learn MCP Server** is publicly hosted at `https://learn.microsoft.com/api/mcp`
and requires no authentication or project connection.

### Key Parameters
| Parameter | Value | Description |
|-----------|-------|-------------|
| `server_label` | `microsoft-learn` | Unique identifier for this MCP server within the agent |
| `server_url` | `https://learn.microsoft.com/api/mcp` | Public MCP endpoint (Streamable HTTP) |
| `require_approval` | `never` | Read-only docs server — no approval needed |

Because this is a read-only public server with no ability to modify resources,
we set `require_approval="never"` to keep the demo flowing smoothly.
For servers that can write or modify data, always use `require_approval="always"`.

In [ ]:
# Define Microsoft Learn MCP Server tool configuration
print("🔌 Configuring Microsoft Learn MCP Server tool...")
print("")

mcp_tool = MCPTool(
    server_label="microsoft-learn",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
)

print(f"✅ Microsoft Learn MCP Tool configured:")
print(f"   Server Label    : {mcp_tool.server_label}")
print(f"   Server URL      : {mcp_tool.server_url}")
print(f"   Require Approval: {mcp_tool.require_approval}")
print(f"   Authentication  : None (public server)")

## 4. Create an Agent with MCP Tools

Now we'll create an agent that uses the Microsoft Learn MCP Server to answer questions about
Azure AI, Microsoft Foundry, and AI concepts — all backed by official, up-to-date Microsoft documentation.

In [ ]:
# Define agent instructions
agent_name = "learn-mcp-assistant"

agent_instructions = """
You are an AI Technology Advisor with access to official Microsoft documentation
via the Microsoft Learn MCP Server.

Your capabilities include:
1. 📚 Searching and retrieving up-to-date Microsoft documentation
2. 🤖 Explaining Microsoft Foundry models, capabilities, and services
3. 🧠 Clarifying AI and machine learning concepts with authoritative sources
4. 🏦 Advising on Azure AI services relevant to financial services workloads
5. 💻 Finding and explaining code samples from Microsoft documentation

When answering questions:
- Always search the Microsoft Learn documentation first using the available MCP tools
- Cite specific documentation pages where relevant
- Provide clear, accurate answers grounded in official Microsoft content
- For model comparisons, reference actual documented capabilities
"""

print(f"📝 Agent Name: {agent_name}")
print(f"📋 Instructions length: {len(agent_instructions)} characters")

In [ ]:
# Create the agent with the Microsoft Learn MCP tool
print(f"Creating Microsoft Learn MCP Assistant Agent...")

agent_definition = PromptAgentDefinition(
    model=model_deployment,
    instructions=agent_instructions,
    tools=[mcp_tool],
)

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=agent_definition,
)

print(f"✅ Agent created: {agent.name} (Version: {agent.version})")
print("🔌 Agent configured with Microsoft Learn MCP Server")
print("")
print("🎯 This agent can now access:")
print("   • microsoft_docs_search       — semantic search across all Microsoft docs")
print("   • microsoft_docs_fetch        — retrieve complete documentation articles")
print("   • microsoft_code_sample_search — search official code samples")

## 5. Create a Conversation

Create a conversation thread to maintain context across multiple interactions with the agent.

In [ ]:
# Create a conversation for the chat session
print("Creating conversation for the chat session...")

conversation = openai_client.conversations.create()

print(f"✅ Conversation created: {conversation.id}")

## 6. Query the Agent with the Learn MCP Tools

The agent will call `microsoft_docs_search` and `microsoft_docs_fetch` transparently to ground its
answers in official Microsoft documentation. Because we set `require_approval="never"`, tool calls
proceed automatically.

The `process_mcp_response` helper below handles any approval requests in the response output —
useful when you later connect to servers that do require approval.

In [ ]:
def process_mcp_response(openai_client, agent, initial_response, auto_approve=True):
    """
    Process a response that may contain MCP approval requests.

    Args:
        openai_client: The OpenAI client instance
        agent: The agent being used
        initial_response: The initial response from the agent
        auto_approve: Whether to automatically approve MCP requests

    Returns:
        The final response after processing any approvals
    """
    response = initial_response

    approval_requests = []
    for item in response.output:
        if item.type == "mcp_approval_request":
            print(f"\n🔐 MCP Approval Request Received:")
            print(f"   Server: {item.server_label}")
            print(f"   Request ID: {item.id}")

            if auto_approve and item.id:
                print(f"   ✅ Auto-approving request...")
                approval_requests.append(
                    McpApprovalResponse(
                        type="mcp_approval_response",
                        approve=True,
                        approval_request_id=item.id,
                    )
                )
            else:
                print(f"   ❌ Request denied or missing ID")

    if approval_requests:
        print(f"\n📤 Sending {len(approval_requests)} approval response(s)...")
        response = openai_client.responses.create(
            input=approval_requests,
            previous_response_id=response.id,
            extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
        )
        print("✅ Approval response sent")

    return response

print("✅ MCP response processing function defined")

In [ ]:
# Sample queries that leverage the Microsoft Learn MCP Server
queries = [
    "What AI models are available in Microsoft Foundry and what are they best suited for?",
    "How do small language models differ from large language models, and when should I choose one over the other?",
    "What is the Model Context Protocol and how does it help AI agents connect to external tools?",
]

print("💬 Microsoft Learn MCP Assistant Session")
print("=" * 60)

for i, query in enumerate(queries, 1):
    print(f"\n📝 Query {i}: {query}")
    print("-" * 50)

    response = openai_client.responses.create(
        conversation=conversation.id,
        input=query,
        extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    )

    response = process_mcp_response(openai_client, agent, response)

    print(f"\n🤖 Agent Response:")
    print(response.output_text if response.output_text else "No response text")
    print("\n" + "=" * 60)

## 7. Interactive Query (Optional)

Try your own questions — the agent will search Microsoft Learn documentation to answer them.

Example prompts to try:
- `"What Azure AI services are available for financial services workloads?"`
- `"How does Azure AI Search support vector search and RAG patterns?"`
- `"What is the difference between Azure OpenAI and Azure AI Foundry?"`
- `"Show me how to use DefaultAzureCredential in Python"`

In [ ]:
custom_query = "What Azure AI services are available for financial services workloads?"

print(f"📝 Custom Query: {custom_query}")
print("-" * 50)

response = openai_client.responses.create(
    conversation=conversation.id,
    input=custom_query,
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
)

response = process_mcp_response(openai_client, agent, response)

print(f"\n🤖 Agent Response:")
print(response.output_text if response.output_text else "No response text")

## 8. Cleanup

Delete the agent when done to clean up resources.

In [ ]:
# Cleanup: Delete the agent
print("🧹 Cleaning up resources...")

try:
    project_client.agents.delete_version(
        agent_name=agent.name,
        agent_version=agent.version
    )
    print(f"✅ Agent deleted: {agent.name} (Version: {agent.version})")
except Exception as e:
    print(f"⚠️ Error during cleanup: {e}")

print("\n🎉 MCP Tools notebook completed!")

## 📚 Summary

In this notebook, we connected an AI agent to the **Microsoft Learn MCP Server** and saw it
ground its answers in official Microsoft documentation.

### What We Did

1. **Configured an MCP tool** pointing to `https://learn.microsoft.com/api/mcp` — no authentication required
2. **Created an agent** with the MCP tool and instructions to search documentation before answering
3. **Queried the agent** about Microsoft Foundry models, SLMs vs LLMs, and MCP itself
4. **Saw MCP in action** — the agent called `microsoft_docs_search` and `microsoft_docs_fetch` transparently

### Key Concepts

| Concept | Detail |
|---------|--------|
| **MCP Transport** | Streamable HTTP — the modern MCP standard |
| **Authentication** | None for public servers; API key or OAuth for private servers |
| **`require_approval`** | `never` for read-only servers; `always` for servers that modify data |
| **`project_connection_id`** | Only needed when the server requires credentials stored in Foundry |

### 🚀 Next Steps

- Connect to a **private MCP server** backed by your own Azure AI Search knowledge base
- Combine **multiple MCP servers** in a single agent for richer tool coverage
- Explore the [MCP specification](https://modelcontextprotocol.io/) to build your own MCP server

### 📚 Resources

- [Microsoft Learn MCP Server Overview](https://learn.microsoft.com/en-us/training/support/mcp)
- [Microsoft Learn MCP Developer Reference](https://learn.microsoft.com/en-us/training/support/mcp-developer-reference)
- [Connect to MCP Endpoints — Microsoft Foundry](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/model-context-protocol)
- [Model Context Protocol Specification](https://modelcontextprotocol.io/)